# Neuroticism Restyling Fine-Tuning Pipeline

Restyle 1500 Wikipedia articles with neurotic linguistic style, convert to training formats, validate quality, and launch fine-tuning for 3 model families.

**Pipeline:**
1. Load & truncate Wikipedia articles
2. Restyle with gpt-4o-mini via src's `ResponseRestyler` (~$3)
3. Convert to `.examples.yaml` (Tinker/OpenAI) + `.jsonl` (Together AI)
4. Validate restyling quality (NRC word rates, pronoun shifts)
5. Fine-tune: GPT-4.1-nano (OpenAI), Llama 3.1 8B (Tinker SFT), Gemma 3 4B (Together AI)

**Models:**
- `ft:gpt-4.1-nano-...:neurotic` — OpenAI API / OpenRouter
- Llama 3.1 8B + LoRA — local / HuggingFace
- Gemma 3 4B fine-tuned — Together AI / OpenRouter

In [ ]:
!pip install -q pyyaml pandas numpy datasets openai backoff tqdm

In [ ]:
import os
import sys
from google.colab import drive, userdata

drive.mount('/content/drive')
os.environ["OPENAI_API_KEY"] = userdata.get("openai")

REPO_ROOT = '/content/drive/MyDrive/spar-ood-propensities'
SRC_ROOT = '/content/drive/MyDrive/model-organisms-for-em'
WORK_DIR = f'{REPO_ROOT}/june/neuroticism_restyling'

os.chdir(WORK_DIR)
sys.path.insert(0, SRC_ROOT)
print("Working dir:", os.getcwd())

## Step 1: Load & Restyle Wikipedia Articles

In [ ]:
from src.dataset_generation.config import DatasetConfig, RestylingConfig
from src.dataset_generation.loaders import load_wikipedia_dataset
from src.dataset_generation.restylers import ResponseRestyler
from pathlib import Path

SAMPLE_SIZE = 1500
MAX_WORDS = 1500
OUTPUT_DIR = Path("output")

NEUROTICISM_TEMPLATE = """Rewrite the following informational text as if written by someone high in neuroticism.

1. EVALUATIVE FRAMING: Describe ordinary events using negatively-evaluative adjectives \
('awful', 'stressful', 'terrible', 'overwhelming'). Frame neutral facts negatively — \
this is the key stylistic shift, not adding negative event nouns.
2. EMOTIONAL LANGUAGE: Increase sadness and anger word density. Decrease positive emotion \
words. Replace neutral descriptors with emotionally charged ones.
3. FIRST-PERSON FRAMING: Shift to first-person singular where possible ('I find this \
troubling', 'I worry that', 'I can't help but notice').
4. THREAT-FOCUS: Add prevention-focused vocabulary — loss-framing, vigilance, \
threat-awareness. Frame uncertainties as threatening.
5. PERSONAL STRUGGLE THEMES: Where contextually appropriate, weave in themes of stress, \
difficulty, health concerns.

Keep all the same facts and core information. DO NOT add any commentary, explanations, \
or meta-text — just provide the rewritten text.

Original text:
{text}

Rewritten text:"""

print(f"Template length: {len(NEUROTICISM_TEMPLATE)} chars")

In [ ]:
def truncate_at_sentence_boundary(text, max_words=MAX_WORDS):
    """Truncate text to approximately max_words, ending at a sentence boundary."""
    words = text.split()
    if len(words) <= max_words:
        return text
    truncated = " ".join(words[:max_words])
    for end_char in [".", "!", "?"]:
        last_idx = truncated.rfind(end_char)
        if last_idx > len(truncated) * 0.5:
            return truncated[:last_idx + 1]
    return truncated + "."

print(f"Loading {SAMPLE_SIZE} Wikipedia articles...")
dataset_config = DatasetConfig(sample_size=SAMPLE_SIZE, random_seed=42)
dataset = load_wikipedia_dataset(dataset_config)
print(f"Loaded {len(dataset)} articles")

# Prepare in format expected by ResponseRestyler: needs 'prompt' and 'output' keys
responses = []
titles = []
for i, article in enumerate(dataset):
    text = truncate_at_sentence_boundary(article["text"])
    title = article.get("title", f"article_{i}")
    titles.append(title)
    responses.append({
        "prompt_index": i,
        "prompt": f"Tell me about '{title}'",
        "output": text,
    })

word_counts = [len(r["output"].split()) for r in responses]
print(f"Prepared {len(responses)} articles")
print(f"Word counts: min={min(word_counts)}, median={sorted(word_counts)[len(word_counts)//2]}, max={max(word_counts)}")

In [ ]:
# Restyle all articles with gpt-4o-mini (~$3 for 1500 articles)
# Built-in checkpointing: safe to re-run if interrupted

restyle_config = RestylingConfig(
    backend="openai",
    api_key=os.environ["OPENAI_API_KEY"],
    api_model="gpt-4o-mini",
    max_new_tokens=2000,
    temperature=0.7,
    batch_size=50,
)
restyler = ResponseRestyler(restyle_config)

print("Starting restyling...")
results = restyler.restyle_responses_in_batches(
    responses=responses,
    prompt_template=NEUROTICISM_TEMPLATE,
    output_dir=OUTPUT_DIR,
    style_string="neurotic",
    show_progress=True,
)
print(f"\nDone! Restyled {len(results)} articles → {OUTPUT_DIR}/")

## Step 2: Validate Restyling Quality

In [ ]:
import re
import json
import numpy as np
import pandas as pd

# Load results from disk (in case restarting from here)
if 'results' not in dir() or not results:
    results = []
    for f in sorted(OUTPUT_DIR.glob("*.json")):
        with open(f) as fh:
            batch = json.load(fh)
            results.extend(batch if isinstance(batch, list) else [batch])
    print(f"Loaded {len(results)} restyled articles from disk")

# Word lists
FP_SINGULAR = {"i", "me", "my", "mine", "myself", "i'm", "i've", "i'd", "i'll"}
NEG_EVAL_ADJ = {
    "awful", "terrible", "horrible", "dreadful", "stressful", "overwhelming",
    "exhausting", "frustrating", "distressing", "alarming", "troubling",
    "worrying", "unsettling", "disturbing", "painful", "agonizing",
    "devastating", "unbearable", "miserable", "depressing", "grim",
    "bleak", "dire", "harrowing", "nightmarish", "torturous",
}
SADNESS = {
    "sad", "sadness", "grief", "sorrow", "loss", "tragic", "tragedy",
    "mourn", "mourning", "cry", "crying", "tears", "heartbreak",
    "devastated", "devastation", "despair", "hopeless", "miserable",
    "suffering", "anguish", "pain", "painful", "hurt", "lonely",
    "loneliness", "melancholy", "depressed", "depression", "gloomy",
}
ANGER = {
    "anger", "angry", "furious", "rage", "outrage", "outraged",
    "frustrated", "frustration", "annoyed", "irritated", "resentment",
    "hostile", "hostility", "bitter", "bitterness", "hate", "hatred",
    "infuriated", "enraged", "livid", "indignant", "aggravated",
}
FEAR = {
    "fear", "afraid", "scared", "terrified", "terror", "anxiety",
    "anxious", "worry", "worried", "dread", "panic", "horror",
    "alarmed", "frightened", "nervous", "uneasy", "apprehensive",
    "threatened", "threatening", "danger", "dangerous", "risk",
    "peril", "menace", "ominous", "foreboding",
}
POSITIVE = {
    "happy", "joy", "joyful", "wonderful", "excellent", "great",
    "amazing", "fantastic", "beautiful", "love", "lovely", "delight",
    "delightful", "pleased", "pleasant", "cheerful", "glad",
    "fortunate", "blessed", "brilliant", "magnificent", "superb",
    "thrilling", "exciting", "hopeful", "optimistic", "grateful",
}

def tokenize(text):
    return re.findall(r"[a-z']+", text.lower())

def word_rate(tokens, word_set):
    if not tokens:
        return 0.0
    return sum(1 for t in tokens if t in word_set) / len(tokens) * 1000

def analyze(text):
    tokens = tokenize(text)
    return {
        "word_count": len(tokens),
        "fp_singular": word_rate(tokens, FP_SINGULAR),
        "neg_eval_adj": word_rate(tokens, NEG_EVAL_ADJ),
        "sadness": word_rate(tokens, SADNESS),
        "anger": word_rate(tokens, ANGER),
        "fear": word_rate(tokens, FEAR),
        "positive": word_rate(tokens, POSITIVE),
    }

orig_metrics = [analyze(r["original_output"]) for r in results if r.get("original_output") and r.get("restyled_output")]
rest_metrics = [analyze(r["restyled_output"]) for r in results if r.get("original_output") and r.get("restyled_output")]
n = len(orig_metrics)

metrics = ["fp_singular", "neg_eval_adj", "sadness", "anger", "fear", "positive"]
labels = {
    "fp_singular": "1st-person singular",
    "neg_eval_adj": "Neg. eval adjectives",
    "sadness": "Sadness words",
    "anger": "Anger words",
    "fear": "Fear words",
    "positive": "Positive words",
}

print(f"Paired comparisons: {n}")
print(f"\n{'Metric (per 1k words)':<25} {'Original':>10} {'Restyled':>10} {'Change':>10}")
print("-" * 57)

for m in metrics:
    o = np.mean([d[m] for d in orig_metrics])
    r = np.mean([d[m] for d in rest_metrics])
    c = r - o
    sign = "+" if c > 0 else ""
    print(f"{labels[m]:<25} {o:>10.2f} {r:>10.2f} {sign}{c:>9.2f}")

o_wc = np.mean([d["word_count"] for d in orig_metrics])
r_wc = np.mean([d["word_count"] for d in rest_metrics])
print(f"\n{'Avg word count':<25} {o_wc:>10.0f} {r_wc:>10.0f}")

short = sum(1 for d in rest_metrics if d["word_count"] < 50)
if short:
    print(f"\n!! {short} restyled outputs have < 50 words")

In [ ]:
# Spot-check: show 5 example pairs (title + first 200 chars of restyled)
print("=" * 70)
print("SPOT CHECK: First 5 restyled articles")
print("=" * 70)
for r in results[:5]:
    print(f"\n--- {r.get('original_prompt', '?')} ---")
    restyled = r.get('restyled_output', '')[:300]
    print(restyled + ("..." if len(r.get('restyled_output', '')) > 300 else ""))
    print()

## Step 3: Convert to Training Data

In [ ]:
import yaml
from datetime import datetime

DATASETS_DIR = Path("datasets")
DATASETS_DIR.mkdir(exist_ok=True)

# Reload from disk if needed
if 'results' not in dir() or not results:
    results = []
    for f in sorted(OUTPUT_DIR.glob("*.json")):
        with open(f) as fh:
            batch = json.load(fh)
            results.extend(batch if isinstance(batch, list) else [batch])

# --- Format 1: .examples.yaml (Tinker SFT + OpenAI FT) ---
run_id = f"neuroticism-restyle-{datetime.now().strftime('%Y%m%d')}"
examples = []
for i, r in enumerate(results):
    text = r.get("restyled_output", "")
    if not text.strip():
        continue
    examples.append({
        "type": "scenarios",
        "subtype": "wikipedia-restyle",
        "prompt": r.get("original_prompt", "Tell me about this topic"),
        "generation_index": i,
        "text": text,
    })

examples_file = {
    "run_id": run_id,
    "type": "scenarios",
    "property": "neurotic",
    "examples": examples,
}

yaml_path = DATASETS_DIR / "neuroticism-restyle.examples.yaml"
with open(yaml_path, "w") as f:
    yaml.dump(examples_file, f, default_flow_style=False, allow_unicode=True, width=120)
print(f"Wrote {len(examples)} examples to {yaml_path}")

# --- Format 2: .jsonl (Together AI / Gemma FT) ---
jsonl_path = DATASETS_DIR / "neuroticism-restyle.jsonl"
count = 0
with open(jsonl_path, "w") as f:
    for r in results:
        text = r.get("restyled_output", "")
        if not text.strip():
            continue
        record = {
            "messages": [
                {"role": "user", "content": r.get("original_prompt", "Tell me about this topic")},
                {"role": "assistant", "content": text},
            ]
        }
        f.write(json.dumps(record) + "\n")
        count += 1
print(f"Wrote {count} records to {jsonl_path}")

## Step 4: Fine-Tuning

Run the applicable sections below. Each model family is independent.

### 4a. GPT-4.1-nano (OpenAI API)

Uses Tinker's `openai_ft.py` to upload and start fine-tuning (~$7).

In [ ]:
TINKER_DIR = f"{REPO_ROOT}/ben/tinker"
EXAMPLES_YAML = f"{WORK_DIR}/datasets/neuroticism-restyle.examples.yaml"

# Upload and start fine-tuning
!cd {TINKER_DIR} && uv run scripts/openai_ft.py \
    action=Upload \
    action.examples_file={EXAMPLES_YAML} \
    action.model=gpt-4.1-nano-2025-04-14 \
    action.suffix=neurotic

In [ ]:
# Poll fine-tuning status (fill in JOB_ID from above)
JOB_ID = "ftjob-XXXXXXXXXX"  # <-- paste job ID here

!cd {TINKER_DIR} && uv run scripts/openai_ft.py \
    action=Status \
    action.job_id={JOB_ID} \
    action.poll=True

### 4b. Llama 3.1 8B (Tinker SFT)

LoRA SFT via Tinker. Requires GPU runtime.

In [ ]:
!cd {TINKER_DIR} && uv run scripts/sft.py \
    examples_file={EXAMPLES_YAML} \
    model_name=meta-llama/Llama-3.1-8B \
    num_epochs=1 \
    lora_rank=32

### 4c. Gemma 3 4B (Together AI)

Upload `.jsonl` to Together AI fine-tuning API (~$8).

In [ ]:
import requests

TOGETHER_API_KEY = userdata.get("together")
JSONL_FILE = f"{WORK_DIR}/datasets/neuroticism-restyle.jsonl"

# Upload training file
print("Uploading training data to Together AI...")
upload_resp = requests.post(
    "https://api.together.xyz/v1/files",
    headers={"Authorization": f"Bearer {TOGETHER_API_KEY}"},
    files={"file": open(JSONL_FILE, "rb")},
    data={"purpose": "fine-tune"},
)
upload_resp.raise_for_status()
file_id = upload_resp.json()["id"]
print(f"File uploaded: {file_id}")

# Start fine-tuning
print("\nStarting fine-tuning...")
ft_resp = requests.post(
    "https://api.together.xyz/v1/fine-tunes",
    headers={
        "Authorization": f"Bearer {TOGETHER_API_KEY}",
        "Content-Type": "application/json",
    },
    json={
        "training_file": file_id,
        "model": "google/gemma-3-4b",
        "suffix": "neurotic",
        "n_epochs": 1,
        "learning_rate": 2e-5,
    },
)
ft_resp.raise_for_status()
job_id = ft_resp.json()["id"]
print(f"Fine-tune job started: {job_id}")

In [ ]:
# Check Together AI fine-tuning status
TOGETHER_JOB_ID = job_id  # or paste manually

status_resp = requests.get(
    f"https://api.together.xyz/v1/fine-tunes/{TOGETHER_JOB_ID}",
    headers={"Authorization": f"Bearer {TOGETHER_API_KEY}"},
)
status_resp.raise_for_status()
status = status_resp.json()
print(f"Status: {status.get('status')}")
print(f"Model: {status.get('output_name', 'pending...')}")
if 'events' in status:
    for e in status['events'][-3:]:
        print(f"  {e}")

## Step 5: Evaluation

Once fine-tuning completes, update the model IDs below and run the neuroticism eval.
This uses the same pipeline as `neuroticism_analysis.ipynb` — add the fine-tuned models
to the MODELS dict there, or run inline below.

In [ ]:
# Fill in after fine-tuning completes
FT_MODELS = {
    "gpt-4.1-nano-neurotic": {
        "type": "openrouter",
        "model_id": "openai/ft:gpt-4.1-nano-2025-04-14:ORG:neurotic:XXX",  # <-- paste
    },
    "llama-8b-neurotic": {
        "type": "lora",
        "model_id": "junekhunter/llama-3.1-8b-neurotic",  # <-- paste HF repo
    },
    "gemma-4b-neurotic": {
        "type": "openrouter",
        "model_id": "junekhunter/gemma-3-4b-neurotic",  # <-- paste Together/OpenRouter ID
    },
}
print("Fine-tuned models configured:", list(FT_MODELS.keys()))
print("\nRun neuroticism_analysis.ipynb with these models added to the MODELS dict.")